# Problem Set 2 - N-Gram Model Construction

Name: **Jessica Mae T. Lanuzo**                | Section: **BSCS 3A**
<hr>

### Dataset used: House MD Transcripts (Season 1)

Description from Kaggle:
- House M.D is an American television medical drama that originally ran on the Fox network for eight seasons, from November 16, 2004 to May 21, 2012. The show's main character is Dr. Gregory House, a pain medication-dependent, unconventional, misanthropic medical genius who leads a team of diagnostic fellows at the fictional Princeton–Plainsboro Teaching Hospital (PPTH) in Princeton, New Jersey.

Available at: https://www.kaggle.com/datasets/kunalbhar/house-md-transcripts/data

# Implementation

## I. Preprocessing

### Setting up libraries

In [1]:
!pip3 install pandas nltk

In [2]:
import pandas as pd
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from itertools import chain

from collections import Counter

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jessica\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Importing the dataset

In [3]:
pd.set_option('display.max_colwidth', None)

In [4]:
# Import dataset
df = pd.read_csv('season1.csv', encoding="latin1")

# Preview first 5 instances
df

,name,line
0,Melanie,Why are you late?
1,Rebecca,You're not going to like the answer.
2,Melanie,I already know the answer.
3,Rebecca,I missed the bus.
4,Melanie,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did."
...,...,...
9791,House,"[loudly] I will not have sex with you! Not again! Miserable, that first time. All that desperate, administrative need """
9792,Cuddy,"Stacy's husband is going to need close monitoring at the hospital. And since we can definitely use her back here, I've offered her a job. General Counsel."
9793,House,Did she say yes?
9794,Cuddy,"She said only if it was okay with you. [House starts to walk off as The Rolling Stones' ""You Can't Always Get What You Want"" plays ironically in the background.] Yes or no?"


### Preparing Dataset

#### Removing bracketed stage directions

In [5]:
# Remove stage directions (text within brackets)
df['line'] = df['line'].apply(
    lambda x: re.sub(r'\[.*?\]', '', str(x))
    )

df

,name,line
0,Melanie,Why are you late?
1,Rebecca,You're not going to like the answer.
2,Melanie,I already know the answer.
3,Rebecca,I missed the bus.
4,Melanie,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did."
...,...,...
9791,House,"I will not have sex with you! Not again! Miserable, that first time. All that desperate, administrative need """
9792,Cuddy,"Stacy's husband is going to need close monitoring at the hospital. And since we can definitely use her back here, I've offered her a job. General Counsel."
9793,House,Did she say yes?
9794,Cuddy,She said only if it was okay with you. Yes or no?


#### Removing "name" column 

In [6]:
df = df.drop(columns=['name'])
# Preview first 5 instances
df.head()

,line
0,Why are you late?
1,You're not going to like the answer.
2,I already know the answer.
3,I missed the bus.
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did."


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9796 entries, 0 to 9795
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   line    9796 non-null   object
dtypes: object(1)
memory usage: 76.7+ KB


In [8]:
num_docs = len(df)
total_words = df['line'].apply(lambda x: len(str(x).split())).sum()

print("Number of documents:", num_docs)
print("Total words:", total_words)

Number of documents: 9796
Total words: 108234


### Text Normalization

In [9]:
# Convert text to lowercase
df['normalized'] = df['line'].apply(lambda x: str(x).lower())

# Preview first 5 instances
df.head()

,line,normalized
0,Why are you late?,why are you late?
1,You're not going to like the answer.,you're not going to like the answer.
2,I already know the answer.,i already know the answer.
3,I missed the bus.,i missed the bus.
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did.","i don't doubt it, no bus stops near brad's. you spent the night, the alarm didn't work. or maybe it did."


### Sentence Tokenization

In [10]:
df['sentences'] = df['normalized'].apply(sent_tokenize)

# Preview first 5 instances
df.head()

,line,normalized,sentences
0,Why are you late?,why are you late?,[ why are you late?]
1,You're not going to like the answer.,you're not going to like the answer.,[ you're not going to like the answer.]
2,I already know the answer.,i already know the answer.,[ i already know the answer.]
3,I missed the bus.,i missed the bus.,[ i missed the bus.]
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did.","i don't doubt it, no bus stops near brad's. you spent the night, the alarm didn't work. or maybe it did.","[ i don't doubt it, no bus stops near brad's., you spent the night, the alarm didn't work., or maybe it did.]"


### Removal of punctuation and special characters

In [11]:
def clean_sentence(sentence):
    sentence = re.sub(r"[^a-z0-9\s]", "", sentence)  # remove punctuation
    sentence = re.sub(r"\s+", " ", sentence).strip() # remove extra spaces
    return sentence

df['clean_sentences'] = df['sentences'].apply(
    lambda s_list: [clean_sentence(s) for s in s_list]
)

# Preview first 5 instances
df.head()

,line,normalized,sentences,clean_sentences
0,Why are you late?,why are you late?,[ why are you late?],[why are you late]
1,You're not going to like the answer.,you're not going to like the answer.,[ you're not going to like the answer.],[youre not going to like the answer]
2,I already know the answer.,i already know the answer.,[ i already know the answer.],[i already know the answer]
3,I missed the bus.,i missed the bus.,[ i missed the bus.],[i missed the bus]
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did.","i don't doubt it, no bus stops near brad's. you spent the night, the alarm didn't work. or maybe it did.","[ i don't doubt it, no bus stops near brad's., you spent the night, the alarm didn't work., or maybe it did.]","[i dont doubt it no bus stops near brads, you spent the night the alarm didnt work, or maybe it did]"


### Addition of Sentence Boundary Tokens

In [12]:
def add_boundaries(sent_list):
    return [f"<s> {s} </s>" for s in sent_list]

df['bounded_sentences'] = df['clean_sentences'].apply(add_boundaries)

# Preview first 5 instances
df.head()


,line,normalized,sentences,clean_sentences,bounded_sentences
0,Why are you late?,why are you late?,[ why are you late?],[why are you late],[<s> why are you late </s>]
1,You're not going to like the answer.,you're not going to like the answer.,[ you're not going to like the answer.],[youre not going to like the answer],[<s> youre not going to like the answer </s>]
2,I already know the answer.,i already know the answer.,[ i already know the answer.],[i already know the answer],[<s> i already know the answer </s>]
3,I missed the bus.,i missed the bus.,[ i missed the bus.],[i missed the bus],[<s> i missed the bus </s>]
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did.","i don't doubt it, no bus stops near brad's. you spent the night, the alarm didn't work. or maybe it did.","[ i don't doubt it, no bus stops near brad's., you spent the night, the alarm didn't work., or maybe it did.]","[i dont doubt it no bus stops near brads, you spent the night the alarm didnt work, or maybe it did]","[<s> i dont doubt it no bus stops near brads </s>, <s> you spent the night the alarm didnt work </s>, <s> or maybe it did </s>]"


### Tokenization into Word Lists

In [13]:
df['tokenized'] = df['bounded_sentences'].apply(
    lambda s_list: [sentence.split() for sentence in s_list]
)
# Preview first 5 instances
df.head()

,line,normalized,sentences,clean_sentences,bounded_sentences,tokenized
0,Why are you late?,why are you late?,[ why are you late?],[why are you late],[<s> why are you late </s>],"[[<s>, why, are, you, late, </s>]]"
1,You're not going to like the answer.,you're not going to like the answer.,[ you're not going to like the answer.],[youre not going to like the answer],[<s> youre not going to like the answer </s>],"[[<s>, youre, not, going, to, like, the, answer, </s>]]"
2,I already know the answer.,i already know the answer.,[ i already know the answer.],[i already know the answer],[<s> i already know the answer </s>],"[[<s>, i, already, know, the, answer, </s>]]"
3,I missed the bus.,i missed the bus.,[ i missed the bus.],[i missed the bus],[<s> i missed the bus </s>],"[[<s>, i, missed, the, bus, </s>]]"
4,"I don't doubt it, no bus stops near Brad's. You spent the night, the alarm didn't work. Or maybe it did.","i don't doubt it, no bus stops near brad's. you spent the night, the alarm didn't work. or maybe it did.","[ i don't doubt it, no bus stops near brad's., you spent the night, the alarm didn't work., or maybe it did.]","[i dont doubt it no bus stops near brads, you spent the night the alarm didnt work, or maybe it did]","[<s> i dont doubt it no bus stops near brads </s>, <s> you spent the night the alarm didnt work </s>, <s> or maybe it did </s>]","[[<s>, i, dont, doubt, it, no, bus, stops, near, brads, </s>], [<s>, you, spent, the, night, the, alarm, didnt, work, </s>], [<s>, or, maybe, it, did, </s>]]"


### Flatten tokens into training corpus

In [14]:
corpus = list(chain.from_iterable(
    chain.from_iterable(df['tokenized'])
))

corpus

['<s>',
 'why',
 'are',
 'you',
 'late',
 '</s>',
 '<s>',
 'youre',
 'not',
 'going',
 'to',
 'like',
 'the',
 'answer',
 '</s>',
 '<s>',
 'i',
 'already',
 'know',
 'the',
 'answer',
 '</s>',
 '<s>',
 'i',
 'missed',
 'the',
 'bus',
 '</s>',
 '<s>',
 'i',
 'dont',
 'doubt',
 'it',
 'no',
 'bus',
 'stops',
 'near',
 'brads',
 '</s>',
 '<s>',
 'you',
 'spent',
 'the',
 'night',
 'the',
 'alarm',
 'didnt',
 'work',
 '</s>',
 '<s>',
 'or',
 'maybe',
 'it',
 'did',
 '</s>',
 '<s>',
 'i',
 'didnt',
 'sleep',
 'with',
 'him',
 '</s>',
 '<s>',
 'girl',
 'theres',
 '</s>',
 '<s>',
 'i',
 'missed',
 'the',
 'bus',
 '</s>',
 '<s>',
 'theres',
 'something',
 'either',
 'very',
 'wrong',
 'with',
 'you',
 'or',
 'theres',
 'something',
 'very',
 'wrong',
 'with',
 'him',
 '</s>',
 '<s>',
 'theres',
 'nothing',
 'wrong',
 'with',
 'him',
 '</s>',
 '<s>',
 'please',
 'tell',
 'me',
 'you',
 'know',
 'that',
 'for',
 'a',
 'fact',
 '</s>',
 '<s>',
 'melanie',
 'i',
 'gotta',
 'go',
 '</s>',
 '<s>',
 

## II. N-Gram Modeling

### Unigram Model

In [15]:
def build_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# Build model
unigrams = build_ngrams(corpus, 1)

In [16]:
# Frequency count
unigram_counts = Counter(unigrams)

unigram_counts

Counter({('<s>',): 16509,
         ('</s>',): 16509,
         ('the',): 3727,
         ('you',): 3426,
         ('to',): 2571,
         ('a',): 2558,
         ('i',): 2346,
         ('it',): 1409,
         ('and',): 1399,
         ('of',): 1223,
         ('is',): 1064,
         ('that',): 1051,
         ('in',): 998,
         ('its',): 941,
         ('not',): 918,
         ('no',): 871,
         ('what',): 854,
         ('your',): 842,
         ('for',): 803,
         ('have',): 763,
         ('we',): 746,
         ('he',): 726,
         ('do',): 703,
         ('on',): 699,
         ('this',): 693,
         ('be',): 675,
         ('im',): 660,
         ('me',): 649,
         ('her',): 646,
         ('if',): 627,
         ('was',): 590,
         ('dont',): 566,
         ('with',): 553,
         ('him',): 550,
         ('youre',): 537,
         ('just',): 535,
         ('but',): 525,
         ('know',): 523,
         ('my',): 522,
         ('so',): 490,
         ('get',): 489,
         (

In [17]:
print("Top 10 Unigrams:", unigram_counts.most_common(10))

Top 10 Unigrams: [(('<s>',), 16509), (('</s>',), 16509), (('the',), 3727), (('you',), 3426), (('to',), 2571), (('a',), 2558), (('i',), 2346), (('it',), 1409), (('and',), 1399), (('of',), 1223)]


### Bigram Model

In [18]:
# Build model
bigrams = build_ngrams(corpus, 2)

In [19]:
# Frequency count
bigram_counts = Counter(bigrams)

bigram_counts

Counter({('</s>', '<s>'): 16508,
         ('<s>', 'i'): 1183,
         ('<s>', 'you'): 876,
         ('<s>', 'no'): 477,
         ('<s>', 'the'): 467,
         ('<s>', 'its'): 461,
         ('<s>', 'what'): 393,
         ('you', '</s>'): 375,
         ('it', '</s>'): 368,
         ('<s>', 'im'): 350,
         ('<s>', 'and'): 324,
         ('<s>', 'well'): 314,
         ('<s>', 'he'): 274,
         ('in', 'the'): 257,
         ('<s>', 'if'): 249,
         ('me', '</s>'): 247,
         ('<s>', 'yeah'): 244,
         ('<s>', 'so'): 237,
         ('<s>', 'we'): 229,
         ('going', 'to'): 220,
         ('<s>', 'oh'): 218,
         ('<s>', 'youre'): 215,
         ('<s>', 'it'): 215,
         ('<s>', 'thats'): 214,
         ('right', '</s>'): 208,
         ('i', 'dont'): 205,
         ('<s>', 'hes'): 198,
         ('that', '</s>'): 193,
         ('him', '</s>'): 189,
         ('<s>', 'but'): 181,
         ('its', 'not'): 176,
         ('of', 'the'): 168,
         ('on', 'the'): 165,
     

In [20]:
# Conditional probabilities
bigram_prob = {
    (w1, w2): count / unigram_counts[(w1,)]
    for (w1, w2), count in bigram_counts.items()
}

bigram_prob

{('<s>', 'why'): 0.008661941971046095,
 ('why', 'are'): 0.09868421052631579,
 ('are', 'you'): 0.2915811088295688,
 ('you', 'late'): 0.0002918855808523059,
 ('late', '</s>'): 0.64,
 ('</s>', '<s>'): 0.9999394269792234,
 ('<s>', 'youre'): 0.013023199466957416,
 ('youre', 'not'): 0.12290502793296089,
 ('not', 'going'): 0.04793028322440087,
 ('going', 'to'): 0.7692307692307693,
 ('to', 'like'): 0.0019447685725398677,
 ('like', 'the'): 0.051771117166212535,
 ('the', 'answer'): 0.0016098738932116983,
 ('answer', '</s>'): 0.5,
 ('<s>', 'i'): 0.07165788357865406,
 ('i', 'already'): 0.002131287297527707,
 ('already', 'know'): 0.08163265306122448,
 ('know', 'the'): 0.040152963671128104,
 ('i', 'missed'): 0.002131287297527707,
 ('missed', 'the'): 0.13333333333333333,
 ('the', 'bus'): 0.002683123155352831,
 ('bus', '</s>'): 0.5384615384615384,
 ('i', 'dont'): 0.08738277919863598,
 ('dont', 'doubt'): 0.0017667844522968198,
 ('doubt', 'it'): 0.3333333333333333,
 ('it', 'no'): 0.0028388928317956,
 ('

In [21]:
print("Top 10 Bigrams:", bigram_counts.most_common(10))

Top 10 Bigrams: [(('</s>', '<s>'), 16508), (('<s>', 'i'), 1183), (('<s>', 'you'), 876), (('<s>', 'no'), 477), (('<s>', 'the'), 467), (('<s>', 'its'), 461), (('<s>', 'what'), 393), (('you', '</s>'), 375), (('it', '</s>'), 368), (('<s>', 'im'), 350)]


### Trigram Model

In [22]:
# Build model
trigrams = build_ngrams(corpus, 3)

In [23]:
# Frequency count
trigram_counts = Counter(trigrams)

trigram_counts

Counter({('</s>', '<s>', 'i'): 1183,
         ('</s>', '<s>', 'you'): 876,
         ('</s>', '<s>', 'no'): 477,
         ('</s>', '<s>', 'the'): 467,
         ('</s>', '<s>', 'its'): 461,
         ('</s>', '<s>', 'what'): 393,
         ('you', '</s>', '<s>'): 375,
         ('it', '</s>', '<s>'): 368,
         ('</s>', '<s>', 'im'): 350,
         ('</s>', '<s>', 'and'): 324,
         ('</s>', '<s>', 'well'): 314,
         ('</s>', '<s>', 'he'): 274,
         ('</s>', '<s>', 'if'): 249,
         ('me', '</s>', '<s>'): 247,
         ('</s>', '<s>', 'yeah'): 244,
         ('</s>', '<s>', 'so'): 237,
         ('</s>', '<s>', 'we'): 229,
         ('</s>', '<s>', 'oh'): 218,
         ('</s>', '<s>', 'youre'): 215,
         ('</s>', '<s>', 'it'): 215,
         ('</s>', '<s>', 'thats'): 214,
         ('right', '</s>', '<s>'): 208,
         ('</s>', '<s>', 'hes'): 198,
         ('that', '</s>', '<s>'): 193,
         ('him', '</s>', '<s>'): 189,
         ('</s>', '<s>', 'but'): 181,
         ('</

In [24]:
# Conditional probabilities for trigrams
trigram_prob = {
    (w1, w2, w3): count / bigram_counts[(w1, w2)]
    for (w1, w2, w3), count in trigram_counts.items()
    if bigram_counts[(w1, w2)] > 0
}

trigram_prob

{('<s>', 'why', 'are'): 0.13986013986013987,
 ('why', 'are', 'you'): 0.8666666666666667,
 ('are', 'you', 'late'): 0.007042253521126761,
 ('you', 'late', '</s>'): 1.0,
 ('late', '</s>', '<s>'): 1.0,
 ('</s>', '<s>', 'youre'): 0.013023988369275503,
 ('<s>', 'youre', 'not'): 0.16744186046511628,
 ('youre', 'not', 'going'): 0.13636363636363635,
 ('not', 'going', 'to'): 0.9090909090909091,
 ('going', 'to', 'like'): 0.01818181818181818,
 ('to', 'like', 'the'): 0.2,
 ('like', 'the', 'answer'): 0.05263157894736842,
 ('the', 'answer', '</s>'): 0.8333333333333334,
 ('answer', '</s>', '<s>'): 1.0,
 ('</s>', '<s>', 'i'): 0.0716622243760601,
 ('<s>', 'i', 'already'): 0.0033812341504649195,
 ('i', 'already', 'know'): 0.2,
 ('already', 'know', 'the'): 0.25,
 ('know', 'the', 'answer'): 0.047619047619047616,
 ('<s>', 'i', 'missed'): 0.0033812341504649195,
 ('i', 'missed', 'the'): 0.4,
 ('missed', 'the', 'bus'): 1.0,
 ('the', 'bus', '</s>'): 0.7,
 ('bus', '</s>', '<s>'): 1.0,
 ('<s>', 'i', 'dont'): 0.11

In [25]:
# Example trigram probability
sample_trigram = ('why', 'are', 'you')
if sample_trigram in trigram_prob:
    print(f"P({sample_trigram[2]} | {sample_trigram[0]} {sample_trigram[1]}) = {trigram_prob[sample_trigram]}")

P(you | why are) = 0.8666666666666667


In [26]:
sample_trigram = ('a', 'chance', 'that')
if sample_trigram in trigram_prob:
    print(f"P({sample_trigram[2]} | {sample_trigram[0]} {sample_trigram[1]}) = {trigram_prob[sample_trigram]}")

P(that | a chance) = 0.08333333333333333


In [27]:
sample_trigram = ('have', 'you', 'seen')
if sample_trigram in trigram_prob:
    print(f"P({sample_trigram[2]} | {sample_trigram[0]} {sample_trigram[1]}) = {trigram_prob[sample_trigram]}")

P(seen | have you) = 0.03125


# Results/Analysis

In [28]:
# Raw vocab (baseline)
# Simple whitespace tokenization from raw "line"
raw_tokens = []
for text in df["line"].astype(str):
    raw_tokens.extend(text.split())

raw_vocab = set(raw_tokens)
raw_vocab_size = len(raw_vocab)

# Final vocab (after preprocessing)
final_vocab = set(corpus)
final_vocab_size = len(final_vocab)

# Reduction percentage
vocab_reduction_pct = ((raw_vocab_size - final_vocab_size) / raw_vocab_size) * 100 if raw_vocab_size else 0

# Sparsity / data-limitation stats
total_tokens = len(corpus)

unique_unigrams = len(unigram_counts)
unique_bigrams = len(bigram_counts)
unique_trigrams = len(trigram_counts)

hapax_unigrams = sum(1 for _, c in unigram_counts.items() if c == 1)
hapax_bigrams = sum(1 for _, c in bigram_counts.items() if c == 1)
hapax_trigrams = sum(1 for _, c in trigram_counts.items() if c == 1)

print("Raw vocab size:", raw_vocab_size)
print("Final vocab size:", final_vocab_size)
print(f"Vocab reduction: {vocab_reduction_pct:.2f}%")

print("Total tokens (final):", total_tokens)
print("Unique unigrams:", unique_unigrams)
print("Unique bigrams:", unique_bigrams)
print("Unique trigrams:", unique_trigrams)

print("Hapax unigrams (count=1):", hapax_unigrams)
print("Hapax bigrams (count=1):", hapax_bigrams)
print("Hapax trigrams (count=1):", hapax_trigrams)

Raw vocab size: 15683
Final vocab size: 8461
Vocab reduction: 46.05%
Total tokens (final): 140795
Unique unigrams: 8461
Unique bigrams: 49911
Unique trigrams: 88375
Hapax unigrams (count=1): 4086
Hapax bigrams (count=1): 36712
Hapax trigrams (count=1): 76070
